In [18]:
import pandas as pd

df = pd.read_excel(r"C:\Users\Bhavan\Desktop\intership Task 3\2019-Nov.xlsx")

print("Shape:", df.shape)
print("\nEvent types:")
print(df['event_type'].value_counts())
print("\nMissing values:")
print(df.isnull().sum())

Shape: (1048575, 9)

Event types:
event_type
view        1014762
purchase      18368
cart          15445
Name: count, dtype: int64

Missing values:
event_time            0
event_type            0
product_id            0
category_id           0
category_code    332077
brand            154315
price                 0
user_id               0
user_session          0
dtype: int64


In [2]:
# Fix datetime
df['event_time'] = pd.to_datetime(df['event_time'], utc=True)
df['event_time'] = df['event_time'].dt.tz_localize(None)

# Extract time columns
df['date'] = df['event_time'].dt.date
df['hour'] = df['event_time'].dt.hour
df['day'] = df['event_time'].dt.day
df['week'] = df['event_time'].dt.isocalendar().week.astype(int)

# Fill missing values
df['category_code'] = df['category_code'].fillna('Unknown')
df['brand'] = df['brand'].fillna('Unknown')

# Extract main category
df['main_category'] = df['category_code'].apply(
    lambda x: x.split('.')[0] if x != 'Unknown' else 'Unknown'
)

print("✅ Cleaning done!")
print("Shape:", df.shape)
print("\nMain categories:")
print(df['main_category'].value_counts())

✅ Cleaning done!
Shape: (1048575, 14)

Main categories:
main_category
electronics     398342
Unknown         332077
appliances      115913
computers        58688
furniture        46631
apparel          40673
auto             17177
construction     16191
kids             13013
accessories       5377
sport             3693
medicine           358
country_yard       284
stationery         158
Name: count, dtype: int64


In [17]:
# Funnel stage counts
total_views = len(df[df['event_type'] == 'view'])
total_carts = len(df[df['event_type'] == 'cart'])
total_purchases = len(df[df['event_type'] == 'purchase'])

view_users = df[df['event_type'] == 'view']['user_id'].nunique()
cart_users = df[df['event_type'] == 'cart']['user_id'].nunique()
purchase_users = df[df['event_type'] == 'purchase']['user_id'].nunique()

print("=" * 45)
print("MARKETING FUNNEL METRICS")
print("=" * 45)
print(f"Stage 1 - AWARENESS (Views):")
print(f"  Unique Users  : {view_users:,}")
print(f"  Total Events  : {total_views:,}")
print()
print(f"Stage 2 - CONSIDERATION (Cart):")
print(f"  Unique Users  : {cart_users:,}")
print(f"  Total Events  : {total_carts:,}")
print()
print(f"Stage 3 - CONVERSION (Purchase):")
print(f"  Unique Users  : {purchase_users:,}")
print(f"  Total Events  : {total_purchases:,}")
print()
print(f"CONVERSION RATES (based on total events):")
print(f"  View → Cart      : {round(total_carts/total_views*100, 2)}%")
print(f"  Cart → Purchase  : {round(total_purchases/total_carts*100, 2)}%")
print(f"  View → Purchase  : {round(total_purchases/total_views*100, 2)}%")
print()
print(f"DROP-OFF RATES:")
print(f"  Views not carted    : {round((1-total_carts/total_views)*100, 2)}%")
print(f"  Views not purchased : {round((1-total_purchases/total_views)*100, 2)}%")

# Revenue
total_revenue = df[df['event_type'] == 'purchase']['price'].sum()
avg_order_value = df[df['event_type'] == 'purchase']['price'].mean()
print()
print(f"REVENUE METRICS:")
print(f"  Total Revenue    : ${total_revenue:,.2f}")
print(f"  Avg Order Value  : ${avg_order_value:,.2f}")

MARKETING FUNNEL METRICS
Stage 1 - AWARENESS (Views):
  Unique Users  : 176,624
  Total Events  : 1,014,762

Stage 2 - CONSIDERATION (Cart):
  Unique Users  : 8,642
  Total Events  : 15,445

Stage 3 - CONVERSION (Purchase):
  Unique Users  : 13,635
  Total Events  : 18,368

CONVERSION RATES (based on total events):
  View → Cart      : 1.52%
  Cart → Purchase  : 118.93%
  View → Purchase  : 1.81%

DROP-OFF RATES:
  Views not carted    : 98.48%
  Views not purchased : 98.19%

REVENUE METRICS:
  Total Revenue    : $5,653,876.49
  Avg Order Value  : $307.81


In [12]:
# Category conversion analysis
category_funnel = df.groupby(['main_category', 'event_type']).agg(
    users=('user_id', 'nunique'),
    events=('event_type', 'count'),
    avg_price=('price', 'mean')
).reset_index()

category_funnel.to_excel('category_funnel.xlsx', index=False)
print("✅ Category funnel saved!")
print(category_funnel.head(10))

✅ Category funnel saved!
  main_category event_type  users  events   avg_price
0       Unknown       cart   1064    1764   96.651094
1       Unknown   purchase   3732    4636  131.019912
2       Unknown       view  69230  325677  177.655028
3   accessories       cart      1       4   20.080000
4   accessories   purchase     29      34   40.503529
5   accessories       view   1391    5339   63.857610
6       apparel       cart      1       2   71.820000
7       apparel   purchase    106     126   88.154048
8       apparel       view   9465   40545   80.601024
9    appliances       cart    400     741  166.379460


In [13]:
# Top brands funnel
top_brands = df[df['event_type'] == 'purchase']['brand'].value_counts().head(20).index.tolist()

brand_funnel = df[df['brand'].isin(top_brands)].groupby(
    ['brand', 'event_type']
).agg(
    events=('event_type', 'count'),
    users=('user_id', 'nunique')
).reset_index()

brand_funnel.to_excel('brand_funnel.xlsx', index=False)
print("✅ Brand funnel saved!")

✅ Brand funnel saved!


In [14]:
# Daily funnel trend
daily_funnel = df.groupby(['day', 'event_type']).agg(
    events=('event_type', 'count'),
    users=('user_id', 'nunique'),
    revenue=('price', 'sum')
).reset_index()

daily_funnel.to_excel('daily_funnel.xlsx', index=False)
print("✅ Daily funnel saved!")

✅ Daily funnel saved!


In [15]:
# Price range analysis
df['price_range'] = pd.cut(df['price'],
    bins=[0, 50, 100, 200, 500, 1000, 99999],
    labels=['$0-50', '$50-100', '$100-200',
            '$200-500', '$500-1000', '$1000+']
)

price_funnel = df.groupby(
    ['price_range', 'event_type'],
    observed=True          # ← this fixes the warning
).agg(
    events=('event_type', 'count'),
    users=('user_id', 'nunique')
).reset_index()

price_funnel.to_excel('price_funnel.xlsx', index=False)
print("✅ Price funnel saved!")
print(price_funnel)

✅ Price funnel saved!
   price_range event_type  events  users
0        $0-50       cart    1463    924
1        $0-50   purchase    3008   2538
2        $0-50       view  181001  49208
3      $50-100       cart    1253    758
4      $50-100   purchase    2231   1906
5      $50-100       view  156624  47477
6     $100-200       cart    4825   2581
7     $100-200   purchase    4795   3809
8     $100-200       view  226178  67107
9     $200-500       cart    4589   2763
10    $200-500   purchase    4931   3967
11    $200-500       view  280176  75285
12   $500-1000       cart    2192   1385
13   $500-1000   purchase    2382   1865
14   $500-1000       view  119285  39914
15      $1000+       cart    1123    725
16      $1000+   purchase    1021    792
17      $1000+       view   50501  20654


In [16]:
# Overall funnel summary
funnel_summary = pd.DataFrame({
    'Stage': ['Awareness (View)', 'Consideration (Cart)', 'Conversion (Purchase)'],
    'Stage_Number': [1, 2, 3],
    'Unique_Users': [views, carts, purchases],
    'Total_Events': [total_views, total_carts, total_purchases],
    'Conversion_Rate': [100,
                        round(carts/views*100, 2),
                        round(purchases/carts*100, 2)],
    'Dropoff_Rate': [0,
                     round((1-carts/views)*100, 2),
                     round((1-purchases/carts)*100, 2)]
})

funnel_summary.to_excel('funnel_summary.xlsx', index=False)
print("✅ Funnel summary saved!")
print(funnel_summary)

✅ Funnel summary saved!
                   Stage  Stage_Number  Unique_Users  Total_Events  \
0       Awareness (View)             1        176624       1014762   
1   Consideration (Cart)             2          8642         15445   
2  Conversion (Purchase)             3         13635         18368   

   Conversion_Rate  Dropoff_Rate  
0           100.00          0.00  
1             4.89         95.11  
2           157.78        -57.78  


In [19]:
# Updated funnel summary with correct numbers
import pandas as pd

funnel_summary = pd.DataFrame({
    'Stage': ['Awareness (View)', 
              'Consideration (Cart)', 
              'Conversion (Purchase)'],
    'Stage_Number': [1, 2, 3],
    'Unique_Users': [176624, 8642, 13635],
    'Total_Events': [1014762, 15445, 18368],
    'Conversion_Rate': [100, 1.52, 1.81],
    'Dropoff_Rate': [0, 98.48, 98.19]
})

funnel_summary.to_excel('funnel_summary.xlsx', index=False)
print("✅ Funnel summary saved!")
print(funnel_summary)

✅ Funnel summary saved!
                   Stage  Stage_Number  Unique_Users  Total_Events  \
0       Awareness (View)             1        176624       1014762   
1   Consideration (Cart)             2          8642         15445   
2  Conversion (Purchase)             3         13635         18368   

   Conversion_Rate  Dropoff_Rate  
0           100.00          0.00  
1             1.52         98.48  
2             1.81         98.19  


In [20]:
import os
files = os.listdir(os.getcwd())
excel_files = [f for f in files if f.endswith('.xlsx')]
print("Excel files in your folder:")
for f in excel_files:
    print(f"✅ {f}")

Excel files in your folder:
✅ 2019-Nov.xlsx
✅ brand_funnel.xlsx
✅ category_funnel.xlsx
✅ daily_funnel.xlsx
✅ funnel_summary.xlsx
✅ price_funnel.xlsx


In [2]:
import os
print("Files in your folder:")
for f in os.listdir(os.getcwd()):
    print(f)

Files in your folder:
.ipynb_checkpoints
2019-Nov.xlsx
brand_funnel.xlsx
category_funnel.xlsx
daily_funnel.xlsx
funnel_summary.xlsx
price_funnel.xlsx
Untitled.ipynb


In [3]:
import pandas as pd

# Load with correct file name
df = pd.read_excel('2019-Nov.xlsx')

# Fix date properly
df['event_time'] = df['event_time'].astype(str)
df['event_time'] = df['event_time'].str.replace(' UTC', '')
df['event_time'] = pd.to_datetime(df['event_time'])

# Extract day
df['day'] = df['event_time'].dt.day

# Check days
print("Unique days:", sorted(df['day'].unique()))
print("Total days:", df['day'].nunique())

Unique days: [np.int32(1)]
Total days: 1


In [4]:
import pandas as pd

df = pd.read_excel('2019-Nov.xlsx')

# Fix date
df['event_time'] = df['event_time'].astype(str)
df['event_time'] = df['event_time'].str.replace(' UTC', '')
df['event_time'] = pd.to_datetime(df['event_time'])

# Extract hour
df['hour'] = df['event_time'].dt.hour

# Check hours
print("Unique hours:", sorted(df['hour'].unique()))

# Create hourly funnel
hourly_funnel = df.groupby(['hour', 'event_type']).agg(
    events=('event_type', 'count'),
    users=('user_id', 'nunique'),
    revenue=('price', 'sum')
).reset_index()

hourly_funnel.to_excel('daily_funnel.xlsx', index=False)
print("✅ Hourly funnel saved!")
print(hourly_funnel.head(10))

Unique hours: [np.int32(0), np.int32(1), np.int32(2), np.int32(3), np.int32(4), np.int32(5), np.int32(6), np.int32(7), np.int32(8), np.int32(9), np.int32(10), np.int32(11), np.int32(12), np.int32(13), np.int32(14), np.int32(15), np.int32(16)]
✅ Hourly funnel saved!
   hour event_type  events  users     revenue
0     0       cart     100     65    44509.08
1     0   purchase     114     88    39277.66
2     0       view   10673   2313  3183232.44
3     1       cart      99     64    34577.93
4     1   purchase     119    106    36209.35
5     1       view   13782   3435  3923589.33
6     2       cart     342    222   103303.23
7     2   purchase     426    352   113181.38
8     2       view   31730   7138  8964843.73
9     3       cart     655    415   182879.45


In [5]:
import pandas as pd

df = pd.read_excel('2019-Nov.xlsx')

# Fix date
df['event_time'] = df['event_time'].astype(str)
df['event_time'] = df['event_time'].str.replace(' UTC', '')
df['event_time'] = pd.to_datetime(df['event_time'])

# Extract BOTH day and hour
df['day'] = df['event_time'].dt.day
df['hour'] = df['event_time'].dt.hour

# Create hourly funnel with BOTH columns
hourly_funnel = df.groupby(['day', 'hour', 'event_type']).agg(
    events=('event_type', 'count'),
    users=('user_id', 'nunique'),
    revenue=('price', 'sum')
).reset_index()

hourly_funnel.to_excel('daily_funnel.xlsx', index=False)
print("✅ Saved with both day and hour!")
print(hourly_funnel.head(10))

✅ Saved with both day and hour!
   day  hour event_type  events  users     revenue
0    1     0       cart     100     65    44509.08
1    1     0   purchase     114     88    39277.66
2    1     0       view   10673   2313  3183232.44
3    1     1       cart      99     64    34577.93
4    1     1   purchase     119    106    36209.35
5    1     1       view   13782   3435  3923589.33
6    1     2       cart     342    222   103303.23
7    1     2   purchase     426    352   113181.38
8    1     2       view   31730   7138  8964843.73
9    1     3       cart     655    415   182879.45
